In [ ]:
from datetime import datetime
from math import ceil


def milliseconds_between(start: datetime, end: datetime) -> int:
    return int(abs((end - start).total_seconds() * 1000))


def ceiled_timeframe_units(start: datetime, end: datetime, timeframe_ms: int) -> int:
    if timeframe_ms <= 0:
        raise ValueError("timeframe_ms must be greater than 0")

    elapsed_ms = milliseconds_between(start, end)
    return ceil(elapsed_ms / timeframe_ms)


In [ ]:
from datetime import datetime, timedelta

start = datetime.now()
end = start + timedelta(milliseconds=1500)

print(milliseconds_between(start, end))          # 1500
print(ceiled_timeframe_units(start, end, 1000))  # 2


In [ ]:
ONE_SECOND = 1000
ONE_MINUTE = 60 * 1000

ceiled_timeframe_units(start, end, ONE_SECOND)
ceiled_timeframe_units(start, end, ONE_MINUTE)


In [ ]:
from datetime import datetime, time, timedelta
from zoneinfo import ZoneInfo

IST = ZoneInfo("Asia/Kolkata")
MARKET_OPEN = time(9, 15)


def normalize_dt(value: datetime) -> datetime:
    if value.tzinfo is None:
        return value.replace(tzinfo=IST)
    return value.astimezone(IST)


def timeframe_delta(timeframe: str) -> timedelta:
    key = timeframe.strip().lower()

    values = {
        "1s": timedelta(seconds=1),
        "1second": timedelta(seconds=1),
        "1m": timedelta(minutes=1),
        "1minute": timedelta(minutes=1),
        "3m": timedelta(minutes=3),
        "5m": timedelta(minutes=5),
        "15m": timedelta(minutes=15),
        "30m": timedelta(minutes=30),
        "1h": timedelta(hours=1),
        "day": timedelta(days=1),
        "1d": timedelta(days=1),
        "week": timedelta(weeks=1),
        "1w": timedelta(weeks=1),
    }

    if key not in values:
        raise ValueError(f"Unsupported timeframe: {timeframe}")

    return values[key]


def candle_bucket(timestamp: datetime, timeframe: str) -> datetime:
    timestamp = normalize_dt(timestamp)
    delta = timeframe_delta(timeframe)

    if timeframe.lower() in {"day", "1d"}:
        return timestamp.replace(hour=9, minute=15, second=0, microsecond=0)

    if timeframe.lower() in {"week", "1w"}:
        monday = timestamp.date() - timedelta(days=timestamp.weekday())
        return datetime.combine(monday, MARKET_OPEN, tzinfo=IST)

    session_start = datetime.combine(timestamp.date(), MARKET_OPEN, tzinfo=IST)
    elapsed = timestamp - session_start

    bucket_number = int(elapsed.total_seconds() // delta.total_seconds())
    return session_start + bucket_number * delta


def is_same_candle(
    latest_candle_timestamp: datetime,
    reading_timestamp: datetime,
    timeframe: str,
) -> bool:
    return candle_bucket(latest_candle_timestamp, timeframe) == candle_bucket(
        reading_timestamp,
        timeframe,
    )
